# Scope definition and data verification

## 1. Imports and parameters

Column names are normalized to uppercase so files from different years can be processed consistently.

In [23]:
from pathlib import Path
import pandas as pd

DATA_FOLDER = Path("dados")
MUNICIPALITIES = ["BRAGA", "GUIMARAES", "GUARDA", "FARO"]


## 2. Reading and combining files

CSV and Excel files are searched for in the `dados` folder and its subfolders. Each file is read separately and the tables are concatenated at the end.

In [24]:

files = sorted(
    file
    for extension in ("xlsx", "xls", "csv")
    for file in DATA_FOLDER.glob(f"*.{extension}")
)

if not files:
    raise FileNotFoundError(f"No files found in folder '{DATA_FOLDER}'.")

tables = []

for file in files:
    print("Reading:", file)

    if file.suffix.lower() == ".csv":
        table = pd.read_csv(
            file,
            sep=None,
            engine="python",
            encoding="utf-8-sig"
        )
    else:
        table = pd.read_excel(file)

    table.columns = table.columns.astype(str).str.upper().str.strip()
    tables.append(table)

df = pd.concat(tables, ignore_index=True)

print(f"Total records: {len(df)}")
print("Columns:", list(df.columns))


Reading: dados\Registos_Incendios_SGIF_2011_2020.csv
Reading: dados\Registos_Incendios_SGIF_2021_2025.csv
Total records: 217735
Columns: ['CODIGO_SGIF', 'CODIGO_ANEPC', 'ANO', 'MES', 'DIA', 'HORA', 'AREAPOV_HA', 'AREAMATO_HA', 'AREAAGRIC_HA', 'AREATOTAL_HA', 'CLASSEAREA', 'DATAHORAALERTA', 'DATAHORA_PRIMEIRAINTERVENCAO', 'DATAHORA_EXTINCAO', 'DURACAO_HORAS', 'INCSUP24HORAS', 'DTCCFR', 'DISTRITO', 'CONCELHO', 'FREGUESIA', 'LOCAL', 'RNAP', 'RNMPF', 'X_MILITAR', 'Y_MILITAR', 'LATITUDE', 'LONGITUDE', 'X_ETRS89', 'Y_ETRS89', 'DSR', 'FWI', 'ISI', 'DC', 'DMC', 'FFMC', 'BUI', 'CODCAUSA', 'TIPOCAUSA', 'GRUPOCAUSA', 'DESCRICAOCAUSA', 'FONTEALERTA']


## 3. Municipality normalization and scope definition

Normalization removes whitespace, converts text to uppercase, and removes accents. Only the four municipalities in the scope are then kept.

In [ ]:
# Standardize municipality names by converting to uppercase, trimming spaces,
# and removing accents/special characters to create a consistent matching key.
df["NORMALIZED_MUNICIPALITY"] = (
    df["CONCELHO"].astype(str).str.upper().str.strip()
    .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
)

# Identify the municipality whose normalized versions
# match the reference municipality scope and display each unique matches.
matching_spellings = df.loc[
    df["NORMALIZED_MUNICIPALITY"].isin(MUNICIPALITIES),
    "CONCELHO"
].unique()
print("Matching spellings:", matching_spellings)

# Filter records belonging to the municipalities scope
# and create an independent copy of the resulting subset.
scoped_data = df[df["NORMALIZED_MUNICIPALITY"].isin(MUNICIPALITIES)].copy()
print(f"Records in scope: {len(scoped_data)}")


Matching spellings: <StringArray>
['Braga', 'Guimarães', 'Guarda', 'Faro']
Length: 4, dtype: str
Records in scope: 5474


## 4. Creating the analysis year

When `DATAALERTA` exists, the year is extracted from that date. Otherwise, the `ANO` column is used.

In [ ]:
# Create a unified year variable from the alert date when available;
# otherwise, use the existing year column.
if "DATAALERTA" in scoped_data.columns:
    scoped_data["ANALYSIS_YEAR"] = pd.to_datetime(
        scoped_data["DATAALERTA"], errors="coerce", dayfirst=True
    ).dt.year
else:
    scoped_data["ANALYSIS_YEAR"] = scoped_data["ANO"]

# Preview the municipality and analysis year fields.
scoped_data[
    [
        "NORMALIZED_MUNICIPALITY",
        "ANALYSIS_YEAR"
    ]
].head() #.head() #.tail()

## 5. Cause normalization

Empty values, `nan`, and `None` are treated as missing information. `MISSING_CAUSE` enables those records to be counted and `NORMALIZED_CAUSE` creates an explicit category.

In [ ]:
# Clean and standardize cause descriptions, flag missing values,
# and assign a default label to records without cause information.
cause = (
    scoped_data["TIPOCAUSA"].astype(str).str.strip()
    .replace({"": None, "nan": None, "None": None})
)
scoped_data["MISSING_CAUSE"] = cause.isna()
scoped_data["NORMALIZED_CAUSE"] = cause.fillna("SEM INFO (vazio)")

# Preview the original, normalized, and missing-cause indicators.
scoped_data[
    [
        "TIPOCAUSA", 
        "NORMALIZED_CAUSE", 
        "MISSING_CAUSE"
    ]
].head()

,TIPOCAUSA,NORMALIZED_CAUSE,MISSING_CAUSE
36,NaN,SEM INFO (vazio),True
132,Negligente,Negligente,False
139,NaN,SEM INFO (vazio),True
140,Negligente,Negligente,False
200,NaN,SEM INFO (vazio),True


## 6. Derived variables for analysis

The time between the alert and the first intervention is calculated, and years are grouped into the periods used by the test.

In [ ]:
alert_time = pd.to_datetime(scoped_data["DATAHORAALERTA"], errors="coerce", dayfirst=True)
intervention_time = pd.to_datetime(
    scoped_data["DATAHORA_PRIMEIRAINTERVENCAO"], errors="coerce", dayfirst=True
)
scoped_data["INTERVENTION_TIME_MINUTES"] = (intervention_time - alert_time).dt.total_seconds() / 60

scoped_data["PERIOD"] = pd.cut(
    scoped_data["ANALYSIS_YEAR"], bins=[2010, 2016, 2017, 2025],
    labels=["2011-2016", "2017", "2018-2025"]
)

scoped_data[["ANALYSIS_YEAR", "INTERVENTION_TIME_MINUTES", "PERIOD"]].head()

,ANALYSIS_YEAR,INTERVENTION_TIME_MINUTES,PERIOD
36,2011,5.0,2011-2016
132,2011,12.0,2011-2016
139,2011,10.0,2011-2016
140,2011,10.0,2011-2016
200,2011,16.0,2011-2016


## 7. Verification tables

These tables reproduce the script checks: records, causes, percentage without a cause, missing values, intervention times, and the final test table.

In [34]:
print("1) Records by year and municipality")
display(pd.crosstab(scoped_data["ANALYSIS_YEAR"], scoped_data["NORMALIZED_MUNICIPALITY"]))

1) Records by year and municipality


NORMALIZED_MUNICIPALITY,BRAGA,FARO,GUARDA,GUIMARAES
ANALYSIS_YEAR,,,,
2011,346,26,43,514
2012,154,34,81,320
2013,184,32,47,272
2014,36,32,35,117
2015,166,44,58,316
2016,135,16,50,240
2017,118,42,69,276
2018,88,26,28,183
2019,82,35,24,119


In [35]:
print("2) Cause type by municipality")
display(pd.crosstab(scoped_data["NORMALIZED_CAUSE"], scoped_data["NORMALIZED_MUNICIPALITY"]))


2) Cause type by municipality


NORMALIZED_MUNICIPALITY,BRAGA,FARO,GUARDA,GUIMARAES
NORMALIZED_CAUSE,,,,
Desconhecida,515,123,39,769
Intencional,122,42,180,191
Natural,0,0,36,0
Negligente,156,233,353,360
Reacendimento,232,0,37,141
SEM INFO (vazio),582,35,4,1324


In [36]:
print("3) % of records without a cause, by year and municipality")
display((pd.crosstab(
    scoped_data["ANALYSIS_YEAR"], scoped_data["NORMALIZED_MUNICIPALITY"], values=scoped_data["MISSING_CAUSE"], aggfunc="mean"
) * 100).round(1))


3) % of records without a cause, by year and municipality


NORMALIZED_MUNICIPALITY,BRAGA,FARO,GUARDA,GUIMARAES
ANALYSIS_YEAR,,,,
2011,80.1,61.5,4.7,91.4
2012,34.4,47.1,0.0,61.6
2013,37.5,0.0,0.0,51.1
2014,11.1,0.0,2.9,1.7
2015,34.9,4.5,0.0,65.2
2016,26.7,0.0,2.0,53.3
2017,32.2,0.0,0.0,64.5
2018,35.2,0.0,0.0,1.6
2019,19.5,0.0,0.0,0.0


In [37]:
columns = [
    "AREATOTAL_HA", "DATAHORAALERTA",
    "DATAHORA_PRIMEIRAINTERVENCAO", "DATAHORA_EXTINCAO", "DURACAO_HORAS",
]
columns = [column for column in columns if column in scoped_data.columns]

print("4) % missing by year")
display((scoped_data.groupby("ANALYSIS_YEAR")[columns].apply(lambda x: x.isna().mean()) * 100).round(1))


4) % missing by year


,AREATOTAL_HA,DATAHORAALERTA,DATAHORA_PRIMEIRAINTERVENCAO,DATAHORA_EXTINCAO,DURACAO_HORAS
ANALYSIS_YEAR,,,,,
2011,0.0,0.0,0.3,0.0,0.0
2012,0.0,0.0,0.0,0.2,0.2
2013,0.0,0.0,0.0,0.0,0.0
2014,0.0,0.0,0.0,0.0,0.0
2015,0.0,0.0,0.0,0.0,0.0
2016,0.0,0.0,0.0,0.0,0.0
2017,0.0,0.0,0.0,0.0,0.0
2018,0.0,0.0,0.0,0.0,0.0
2019,0.0,0.0,0.0,0.0,0.0


In [38]:
print("5) Time to first intervention (minutes)")
display(scoped_data.groupby("NORMALIZED_MUNICIPALITY")["INTERVENTION_TIME_MINUTES"].describe().round(1))
print("Negative:", int((scoped_data["INTERVENTION_TIME_MINUTES"] < 0).sum()),
      "| Above 24h:", int((scoped_data["INTERVENTION_TIME_MINUTES"] > 1440).sum()))


5) Time to first intervention (minutes)


,count,mean,std,min,25%,50%,75%,max
NORMALIZED_MUNICIPALITY,,,,,,,,
BRAGA,1603.0,12.2,8.0,0.0,7.0,11.0,15.0,139.0
FARO,433.0,10.8,5.0,1.0,8.0,10.0,13.0,36.0
GUARDA,643.0,15.0,12.6,0.0,9.0,12.0,19.0,220.0
GUIMARAES,2784.0,12.5,15.0,0.0,7.0,11.0,15.0,615.0


Negative: 0 | Above 24h: 0


In [39]:

hypothesis_data = scoped_data[
    scoped_data["NORMALIZED_CAUSE"].str.lower().isin(["negligente", "intencional"])
    & (scoped_data["PERIOD"] != "2017")
]
print("6) Negligent vs intentional by period (2017 excluded)")
display(pd.crosstab([hypothesis_data["NORMALIZED_MUNICIPALITY"], hypothesis_data["PERIOD"]], hypothesis_data["NORMALIZED_CAUSE"]))


6) Negligent vs intentional by period (2017 excluded)


NORMALIZED_CAUSE                   Intencional  Negligente
NORMALIZED_MUNICIPALITY PERIOD                            
BRAGA                   2011-2016           36         101
                        2018-2025           85          48
FARO                    2011-2016            9          98
                        2018-2025           33         117
GUARDA                  2011-2016           87         172
                        2018-2025           66         147
GUIMARAES               2011-2016           24         136
                        2018-2025          167         221